# Getting started with MemsArrayWS objects

The `MemsArrayWS` class allows getting signals from a remote antenna running a local *Megamicros Broadcast Server (MBS)* server. 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from megamicros.log import log
from megamicros.core.ws import MemsArrayWS

log.setLevel( "INFO" )

# Set server access credentials
HOST = 'buzenval20.fr'
PORT = 9002

## Connecting to the remote server

Providing a *MBS* server is running at ``HOST:PORT``, one can try to connect by creating a ``MemsArrayWS`` object.

In [2]:
# Define the antenna
try:
    antenna = MemsArrayWS( host=HOST, port=PORT )
except Exception as e:
    print( f"Failed: {e}" )


2023-09-27 22:03:14,348 [INFO]:  .Async event loop already running. Adding coroutine to the event loop...


2023-09-27 22:03:14,355 [INFO]:  .Try connecting to ws://buzenval20.fr:9002...
2023-09-27 22:03:14,386 [INFO]:  .Received positive answer from server
2023-09-27 22:03:14,387 [INFO]:  .Getting settings values from remote receiver...
2023-09-27 22:03:14,952 [INFO]:  .Received settings from server [ok]
2023-09-27 22:03:14,953 [INFO]:  .Set 8 MEMs numbered from 0 to 7
2023-09-27 22:03:14,954 [INFO]:  .Set 0 analog channels numbered from 0 to -1
2023-09-27 22:03:14,958 [INFO]:  .Starting MegamicrosWS device [ready]


In [3]:
antenna.setActiveMems( [1, 2] )
antenna.run( duration=0 )

2023-09-27 22:03:17,595 [INFO]:  .2 MEMs were activated among 0 to 7 available MEMs
2023-09-27 22:03:17,598 [INFO]:  .Starting run execution
2023-09-27 22:03:17,599 [INFO]:  .Install run settings
2023-09-27 22:03:17,601 [INFO]:  .Pre-execution checks
2023-09-27 22:03:17,601 [INFO]:  .Counter was not set -> set to False
2023-09-27 22:03:17,602 [INFO]:  .Counter skipping not set -> set to False
2023-09-27 22:03:17,603 [INFO]:  .Data transfer using queue
2023-09-27 22:03:17,604 [INFO]:  .Starting run thread execution...
2023-09-27 22:03:17,605 [INFO]:  .Stop here execution !


In [ ]:
print( 'sf = ', antenna.sampling_frequency )
print( 'available mems = ', antenna.available_mems )
print( 'counter: ', antenna.counter )
print( 'counter skipping: ', antenna.counter_skip )
antenna_definition = np.load ('Antenna-square-JetsonNano-0001.npy', allow_pickle=True )
antenna.setMemsPosition( antenna_definition.item().get('positions') )
antenna.setActiveMems( antenna_definition.item().get('mems'))
print( 'active mems = ', antenna.mems )


## With initialization parameters 

In [ ]:
# Create a 10 MEMs antenna
antenna = MemsArray( available_mems_number=10, mems=[0, 1, 2] )

## With antenna physical parameters

In [ ]:
# Create an antenna from physical parameters
antenna_definition = np.load ('Antenna-square-JetsonNano-0001.npy', allow_pickle=True )
mems_position = antenna_definition.item().get('positions')
antenna = MemsArray( mems_position=mems_position )

# Create antenna and set activated MEMs 
antenna = MemsArray( 
    available_mems_number = len( antenna_definition.item().get('available_mems') ),
    mems_position=mems_position,
    mems=antenna_definition.item().get('mems') 
)

## Plot the antenna MEMs

In [ ]:
mems_position = antenna.mems_position
fig = plt.figure()
ax = fig.add_subplot( 121, projection='3d' )
ax.scatter( mems_position[:,0], mems_position[:,1], mems_position[:,2], marker='^' )

## Generate some signals

Since no input stream is declared, the antenna generates some random samples from a uniform distribution over [-1, +1].
No frame size has been specified either. Default is given by ``megamicros.base.DEFAULT_FRAME_LENGTH`` which is set to 256.

In [ ]:
# iterate over the antenna data stream
for i, data in enumerate( antenna ):
    print( f"data({data.shape})={data}")
    if i > 10:
        break

## Setting a DB input stream

Setting a DB stream as input consists in connecting the Antenna to a Megamicros remote database with the adequat address and request.
In what follows, one ask the data base to send the first sequence from the file number 8692 for which label is 18:

In [ ]:
antenna.setInputDB( label_id = 18, file_id=8692, sequence_id=0 )

## Getting signal from DB

In [ ]:
# Available labels
LABEL_SOW_FEEDING_CALL = 18
LABEL_PIGLET_SQUEALS = 15
LABEL_SOW_GRUNT_NERVOUS = 16
LABEL_ROOM_NOISE = 29
LABEL_SOW_GRUNT = 8
LABEL_SOW_GRUNT_MODSTRESS  = 1
LABEL_SOW_SCREAMS = 3
LABEL_PIGLET_SQUEALS_2 = 5

# choose label, file and sequence in file:
LABEL_ID = LABEL_SOW_FEEDING_CALL
FILE_ID = 8692          # 5838 (1), 7135 (1), 6860(3), 6560(1)
SEQUENCE_ID = 0         

with AidbSession(
    dbhost='http://dbwelfare.biimea.io/',
    login='ailab',
    email='bruno.gas@biimea.com',
    password='#T;uZnQ5UJ_JC~&' ) as session:
        signal: MuAudio = session.load_labelized( 
            sourcefile_id=FILE_ID, 
            label_id=LABEL_ID, 
            limit=100, 
            channels=list( np.arange( 32 ) + 1 ) 
        )[SEQUENCE_ID]

# get infos
LABEL_TXT = signal.label
CHANNELS_NUMBER = signal.channels_number
SAMPLES_NUMBER = signal.samples_number
SAMPLING_FREQUENCY = signal.sampling_frequency

print( f"Some informations about the signal loaded:" )
print( f" > label={LABEL_TXT}" )
print( f" > channels_number={CHANNELS_NUMBER}" )
print( f" > samples_number={SAMPLES_NUMBER}" )
print( f" > sampling_frequency={SAMPLING_FREQUENCY}" )

In [ ]:
# Play sound using channel 0 and 1
left = np.array( signal.channel(0) )
right = np.array( signal.channel(1) )
sound = np.array( [left, right] ).T

display.Audio( sound, rate=SAMPLING_FREQUENCY )

## Set the beamformer

In [ ]:
FRAME_LENGTH = 1024
AREA = [12, 14, 0.01]
AREA_QUANTIZATION = [4, 4, 1/0.01]

# Get the antenna physical description
antenna = np.load ('Antenna-square-JetsonNano-0001.npy', allow_pickle=True )
mems_position = antenna.item().get("positions")

# Create the beamformer
bmf = Beamformer( 
    mems_position = mems_position,
    sampling_frequency = SAMPLING_FREQUENCY,
    window_size = FRAME_LENGTH,    
    area = AREA,
    area_quantization = AREA_QUANTIZATION
)

# Move the antenna in the right place:
bmf.moveArea( [0, 0, -2] )

# Limit the frequency bandwidth for BF computing
bmf.setBandWidth( [200, 2000], unit="frequency" )

# Init the beamformer
bmf.init()

# print area locations and antenna 
space_locations = bmf.getLocations()
mems_location = bmf.getMems()
nx, ny, nz = bmf.getLocationsNumber()

fig = plt.figure()
ax = fig.add_subplot( 121, projection='3d' )
ax.scatter( space_locations[:,0], space_locations[:,1], space_locations[:,2] )
ax.scatter( mems_location[:,0], mems_location[:,1], mems_location[:,2], marker='^' )
ax = fig.add_subplot( 122, projection='3d' )
ax.scatter( mems_location[:,0], mems_location[:,1], mems_location[:,2], marker='^' )
fig.show()


## Compute preformed channels 

In [ ]:
# Get the whole 32 channels signal as a numpy.ndarray
signal32 = signal().T

# Check if some available mems have not been activated
# Remove from signal if any
mems = antenna.item().get('mems')
available_mems = antenna.item().get('available_mems')
if False in np.isin( available_mems, mems ):
    mask = list( np.invert( np.logical_not( np.isin( available_mems, mems ) ) ) )
    signal32 = signal32[:,mask]

FRAMES_NUMBER = SAMPLES_NUMBER // FRAME_LENGTH - 1

print( f"{FRAMES_NUMBER} frames of {FRAME_LENGTH} samples to perform... " )


imgs = []
for i in range( FRAMES_NUMBER ):
    bf = bmf.beamform( signal32[i*FRAME_LENGTH:(i+1)*FRAME_LENGTH,:] )
    imgs.append( np.reshape( bf, (nx, ny) ) )

### Make video

In [ ]:
generate_moovie( 
    imgs, 
    rate=SAMPLING_FREQUENCY/FRAME_LENGTH,
    sound=sound.astype( np.float32 ).T,
    sampling_frequency=SAMPLING_FREQUENCY,
    norm=None,
    extent=( 0, AREA[0], 0, AREA[1] ),
    cleanup=True
)

### Do the same with an antenna object

In [ ]:
# Declare a MEMs antenna
antenna = MemsArray( available_mems_number=CHANNELS_NUMBER )

# set active mems
antenna.setActiveMems( [i for i in range( CHANNELS_NUMBER )] )
print( f"active mems number={antenna.mems_number}" )

# iterate over the antenna data stream
#for i, data in enumerate( antenna ):
#    print( f"data={data}")
#    if i > 10:
#        break
